# Applied Paper Examples — did_multiplegt_stat

**Paper:** de Chaisemartin, d'Haultfoeuille, Pasquier, Vazquez-Bare & Sow (2025)  
**Reference:** Companion paper, Sections 3.1 and 3.2  

This notebook contains the 5 applied examples from the companion paper.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from did_multiplegt_stat import did_multiplegt_stat, summary_did_multiplegt_stat

# Load gasoline data
df = pd.read_stata("gazoline_did_multiplegt_stat.dta")

---

## Applied Paper 1 — Li et al. (2014): Gasoline Taxes and Consumer Behavior

**Reference:** Companion paper, Section 3.1

### Example 1: Reduced Form

**Stata:** `eststo ReducedForm: did_multiplegt_stat lngca id year tau, or(1) as_vs_was controls(lngpinc) cross_fitting(10)`

In [2]:
r_rf = did_multiplegt_stat(df, "lngca", "id", "year", "tau",
                           order=1, aoss_vs_waoss=True,
                           controls=["lngpinc"], cross_fitting=10)
summary_did_multiplegt_stat(r_rf)

                                  ----------------------------------------------
                                   Number of observations     =             1632
                                   Estimation method          =    doubly-robust
                                  Polynomial order           =              (1)
                                  ----------------------------------------------
--------------------------------------------------------------------------------
                               Average Slope (AS)
--------------------------------------------------------------------------------
      Estimate         SE       LB CI       UB CI Switchers Stayers
AS  -0.0058344  0.0027242  -0.0111738  -0.0004949       384   1,248
--------------------------------------------------------------------------------
                          Weighted Average Slope (WAS)
--------------------------------------------------------------------------------
       Estimate         SE     

### Example 2: First Stage

**Stata:** `eststo FirstStage: did_multiplegt_stat lngpinc id year tau, or(1) as_vs_was controls(lngpinc) cross_fitting(10)`

In [3]:
r_fs = did_multiplegt_stat(df, "lngpinc", "id", "year", "tau",
                           order=1, aoss_vs_waoss=True,
                           controls=["lngpinc"], cross_fitting=10)
summary_did_multiplegt_stat(r_fs)

                                  ----------------------------------------------
                                   Number of observations     =             1632
                                   Estimation method          =    doubly-robust
                                  Polynomial order           =              (1)
                                  ----------------------------------------------
--------------------------------------------------------------------------------
                               Average Slope (AS)
--------------------------------------------------------------------------------
     Estimate         SE       LB CI      UB CI Switchers Stayers
AS  0.0033842  0.0024339  -0.0013863  0.0081547       384   1,248
--------------------------------------------------------------------------------
                          Weighted Average Slope (WAS)
--------------------------------------------------------------------------------
      Estimate         SE      LB C

### Example 3: IV Estimation

**Stata:** `eststo IV: did_multiplegt_stat lngca id year lngpinc tau, or(1) controls(lngpinc) cross_fitting(10) bootstrap(5) seed(1)`

In [4]:
r_iv = did_multiplegt_stat(df, "lngca", "id", "year", "lngpinc", Z="tau",
                           estimator="ivwaoss", order=1,
                           controls=["lngpinc"], cross_fitting=10,
                           bootstrap=5, seed=1)
summary_did_multiplegt_stat(r_iv)

                              First stage estimation
                              Reduced form estimation
Bootstrap (5 replications)...
                                  ----------------------------------------------
                                   Number of observations     =               48
                                   IV-WAS Estimation method     =    doubly-robust
                                   IV regression package        =    manual 2SLS
                                  Polynomial order           =              (1)
                                  ----------------------------------------------
--------------------------------------------------------------------------------
                       IV-Weighted Average Slope (IV-WAS)
--------------------------------------------------------------------------------
          Estimate         SE       LB CI      UB CI Switchers Stayers
IV-WAS  -0.6636981  1.7971465  -4.1861053  2.8587091       384   1,248
--------------

### Example 4: On Placebo Sample (robust to dynamic effects)

**Stata (reduced form):** `did_multiplegt_stat lngca id year tau, or(1) controls(lngpinc) estimator(was) cross_fitting(10) on_placebo_sample`  
**Stata (first stage):** `did_multiplegt_stat lngpinc id year tau, or(1) controls(lngpinc) estimator(was) cross_fitting(10) on_placebo_sample`

In [5]:
# Reduced form on placebo sample
r_op1 = did_multiplegt_stat(df, "lngca", "id", "year", "tau",
                            estimator="waoss", order=1,
                            controls=["lngpinc"], cross_fitting=10,
                            on_placebo_sample=True)
summary_did_multiplegt_stat(r_op1)

                                  ----------------------------------------------
                                   Number of observations     =               48
                                   Estimation method          =    doubly-robust
                                  Polynomial order           =              (1)
                                  ----------------------------------------------
--------------------------------------------------------------------------------
                          Weighted Average Slope (WAS)
--------------------------------------------------------------------------------
       Estimate         SE       LB CI       UB CI Switchers Stayers
WAS  -0.0044213  0.0011489  -0.0066731  -0.0021695       178     881
--------------------------------------------------------------------------------


In [6]:
# First stage on placebo sample
r_op2 = did_multiplegt_stat(df, "lngpinc", "id", "year", "tau",
                            estimator="waoss", order=1,
                            controls=["lngpinc"], cross_fitting=10,
                            on_placebo_sample=True)
summary_did_multiplegt_stat(r_op2)

                                  ----------------------------------------------
                                   Number of observations     =               48
                                   Estimation method          =    doubly-robust
                                  Polynomial order           =              (1)
                                  ----------------------------------------------
--------------------------------------------------------------------------------
                          Weighted Average Slope (WAS)
--------------------------------------------------------------------------------
      Estimate         SE      LB CI      UB CI Switchers Stayers
WAS  0.0053249  0.0012069  0.0029593  0.0076905       178     881
--------------------------------------------------------------------------------


---

## Applied Paper 2 — Gentzkow et al. (2011): Newspapers and Electoral Politics

**Reference:** Companion paper, Section 3.2

### Example 5: Exact Match with Placebo

**Stata:** `did_multiplegt_stat prestout cnty90 year numdailies, placebo(1) exact_match`

**Expected from Stata:**
- AS = 0.0060609, SE = 0.0016266
- WAS = 0.0057148, SE = 0.001488
- Placebo AS = -0.0011279
- Placebo WAS = -0.0000125
- Switchers = 4423, Stayers = 11043, N = 15466

In [7]:
df_g = pd.read_stata("gentzkowetal_didtextbook.dta")
r_g = did_multiplegt_stat(df_g, "prestout", "cnty90", "year", "numdailies",
                          placebo=1, exact_match=True)
summary_did_multiplegt_stat(r_g)

                                  ----------------------------------------------
                                   Number of observations     =            15466
                                   Estimation method          =    reg. adjustment
                                  Common support            =  exact matching
                                  ----------------------------------------------
--------------------------------------------------------------------------------
                               Average Slope (AS)
--------------------------------------------------------------------------------
     Estimate         SE      LB CI      UB CI Switchers Stayers
AS  0.0060724  0.0016308  0.0028759  0.0092688     4,423  11,043
--------------------------------------------------------------------------------
                                 Placebo(s) AS
--------------------------------------------------------------------------------
             Estimate         SE       LB CI 